<a href="https://colab.research.google.com/github/densu1306/olist-ecommerce-analytics/blob/main/01_olist_etl_to_bigquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Verify all 9 Olist CSV files are visible
DATA_PATH = '/content/drive/MyDrive/Portfolio_Data/olist/'
files = sorted(os.listdir(DATA_PATH))

print(f'Found {len(files)} files:')
for f in files:
  print(' -', f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 9 files:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


In [ ]:
import pandas as pd
import numpy as np

# 1. Load all 9 CSVs into a dictionary
raw_files = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for name, filename in raw_files.items():
  dfs[name] = pd.read_csv(os.path.join(DATA_PATH, filename))
  print(
      f'Loaded {name:20s} -> {dfs[name].shape[0]:,} rows x'
      f' {dfs[name].shape[1]} cols'
   )

Loaded customers            -> 99,441 rows x 5 cols
Loaded geolocation          -> 1,000,163 rows x 5 cols
Loaded order_items          -> 112,650 rows x 7 cols
Loaded order_payments       -> 103,886 rows x 5 cols
Loaded order_reviews        -> 99,224 rows x 7 cols
Loaded orders               -> 99,441 rows x 8 cols
Loaded products             -> 32,951 rows x 9 cols
Loaded sellers              -> 3,095 rows x 4 cols
Loaded category_translation -> 71 rows x 2 cols


In [ ]:
# Initial Quality Audit Datasets

def print_clean_info(df):
  print(f'\n{'='*25} {df.table_name.upper()} TABLE {'='*25}')
  print(
      f"Rows: {df.shape[0]} | Columns: {df.shape[1]} | Duplicate Rows:"
      f" {df.duplicated().sum()}\n"
  )

  summary = pd.DataFrame({
      "column_name": df.columns,
      "missing_values": df.isna().sum().values,
      "dtype": df.dtypes.values,
  })
  print(summary)


for table_name, df in dfs.items():
  df.table_name = table_name
  print_clean_info(df)


========================= CUSTOMERS TABLE =========================
Rows: 99441 | Columns: 5 | Duplicate Rows: 0

                column_name  missing_values   dtype
0               customer_id               0  object
1        customer_unique_id               0  object
2  customer_zip_code_prefix               0   int64
3             customer_city               0  object
4            customer_state               0  object

========================= GEOLOCATION TABLE =========================
Rows: 1000163 | Columns: 5 | Duplicate Rows: 261831

                   column_name  missing_values    dtype
0  geolocation_zip_code_prefix               0    int64
1              geolocation_lat               0  float64
2              geolocation_lng               0  float64
3             geolocation_city               0   object
4            geolocation_state               0   object

========================= ORDER_ITEMS TABLE =========================
Rows: 112650 | Columns: 7 | Duplicate Rows

In [ ]:
# CLEANING AND TRASNFORMING THE DATA
# A. Customers: Standardize 5-digit zip code prefix with leading zeros
dfs['customers']['customer_zip_code_prefix'] = (
    dfs['customers']['customer_zip_code_prefix'].astype(str).str.zfill(5)
)

# B. Sellers: Standardize 5-digit zip code prefix with leading zeros
dfs['sellers']['seller_zip_code_prefix'] = (
    dfs['sellers']['seller_zip_code_prefix'].astype(str).str.zfill(5)
)

# C. Orders: Parse all 5 timestamp columns & nullify the 24 impossible carrier-after-customer glitches
order_dates = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
dfs['orders'][order_dates] = dfs['orders'][order_dates].apply(
    pd.to_datetime, errors='coerce'
)

invalid_carrier = (
    dfs['orders']['order_delivered_customer_date']
    < dfs['orders']['order_delivered_carrier_date']
)
dfs['orders'].loc[invalid_carrier, 'order_delivered_carrier_date'] = pd.NaT

# D. Order Items: Parse shipping_limit_date
dfs['order_items']['shipping_limit_date'] = pd.to_datetime(
    dfs['order_items']['shipping_limit_date'], errors='coerce'
)

# E. Order Reviews: Parse dates & deduplicate to 1 latest review per order_id (prevents SQL join fan-out)
review_dates = ['review_creation_date', 'review_answer_timestamp']
dfs['order_reviews'][review_dates] = dfs['order_reviews'][review_dates].apply(
    pd.to_datetime, errors='coerce'
)

dfs['order_reviews'] = dfs['order_reviews'].sort_values(
    'review_creation_date', ascending=False
).drop_duplicates(subset=['order_id'], keep='first')

# F. Products: Merge English translations, map the 2 untranslated categories, fill missing with 'unknown'
# Drop the portugesee category name and change it to english name only
dfs['products'] = dfs['products'].merge(
    dfs['category_translation'], on='product_category_name', how='left'
)

missing_translations = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': (
        'kitchen_and_food_processors'
    ),
}

dfs['products']['product_category_name_english'] = (
    dfs['products']['product_category_name_english']
    .fillna(dfs['products']['product_category_name'].map(missing_translations))
    .fillna('unknown')
)

dfs['products'] = dfs['products'].drop(
    columns=['product_category_name']
).rename(columns={'product_category_name_english': 'product_category_name'})

# G. Geolocation: Standardize 5-digit zip, filter out-of-Brazil coordinate outliers, & deduplicate by zip prefix
dfs['geolocation']['geolocation_zip_code_prefix'] = (
    dfs['geolocation']['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
)

in_brazil = (
    dfs['geolocation']['geolocation_lat'].between(-34.0, 5.5)
    &
    dfs['geolocation']['geolocation_lng'].between(-74.5, -34.0)
)

dfs['geolocation'] = (
    dfs['geolocation'][in_brazil]
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': 'first',
        'geolocation_state': 'first',
    })
)

In [ ]:
# CHECK THE CLEANING RESULT
for table_name, df in dfs.items():
  df.table_name = table_name
  print_clean_info(df)

In [ ]:
# PUSHING THE CLEAN TABLE TO GOOGLE BIGQUERY
from google.colab import auth
from pandas_gbq import to_gbq

# 1. Authenticate your Google Cloud account (if not already run at the top)
auth.authenticate_user()

PROJECT_ID = 'data-portfolio-515209'
DATASET_ID = 'olist_ecommerce'

# 2. List the 8 clean tables to upload
clean_tables = [
    'customers',
    'geolocation',
    'order_items',
    'order_payments',
    'order_reviews',
    'orders',
    'products',
    'sellers',
]

# 3. Push each DataFrame to BigQuery
for table_name in clean_tables:
  print(f'Uploading {table_name} ({len(dfs[table_name]):,} rows)...')
  to_gbq(
      dfs[table_name],
      destination_table=f'{DATASET_ID}.{table_name}',
      project_id=PROJECT_ID,
      if_exists='replace',
  )

print('\n✅ All 8 clean tables successfully loaded into BigQuery!')

Uploading customers (99,441 rows)...


100%|██████████| 1/1 [00:00<00:00, 5974.79it/s]


Uploading geolocation (19,010 rows)...


100%|██████████| 1/1 [00:00<00:00, 10727.12it/s]


Uploading order_items (112,650 rows)...


100%|██████████| 1/1 [00:00<00:00, 15087.42it/s]


... (7 outputs left out)

100%|██████████| 1/1 [00:00<00:00, 12018.06it/s]


Uploading sellers (3,095 rows)...


100%|██████████| 1/1 [00:00<00:00, 10727.12it/s]

✅ All 8 clean tables successfully loaded into BigQuery!
